# 05 — landing_validate

## Purpose
Quality gate between bootstrap scripts and Bronze processing.
Validates that all raw files are present, structurally correct, and readable
before any Bronze notebook runs.

**This notebook must run and pass before any Bronze notebook.**

## Behaviour on failure
- Logs the result to `db/bootstrap_log.json` with `source_id = 'landing_validate'`
- Prints a highlighted error message listing every failed check
- Raises `AssertionError` — halts the Jupyter kernel, preventing Bronze from running

## Behaviour on success
- Logs the result to `db/bootstrap_log.json` with `status = 'SUCCESS'`
- Prints a green summary table
- Writes `db/landing_gate.json` — Bronze notebooks verify this file in Step 1

## Input
- `data/raw/receita_federal/{YYYY-MM}/` — 27 zip files per month
- `data/raw/pncp/{YYYY-MM}.json` — one JSON per month  
- `data/raw/portal_transparencia/ceis.json` + `cnep.json` — full historical base
- `data/raw/compras_gov/{YYYY-MM}.json` — one JSON per month
- `db/bootstrap_log.json` — execution log from bootstrap scripts

## Output
- `db/bootstrap_log.json` — appended with landing_validate result
- `db/landing_gate.json` — gate file read by Bronze notebooks

## Pipeline position
```
01 bootstrap_receita_federal
02 bootstrap_pncp
03 bootstrap_transparencia
04 bootstrap_compras
05 landing_validate          ← this notebook
06 bronze_compras
07 bronze_pncp
...
```

## Notes
- Idempotent — safe to re-run. Only reads raw files, never writes to data/.
- On Databricks: replace `raise AssertionError` with `dbutils.notebook.exit()`
  so the DAB orchestrator can control the workflow.
- Future enhancement: add webhook notification before raise (Slack/Teams/email)
  for scheduled pipelines with SLA requirements.


## Step 1 — Imports and configuration

In [ ]:
import json
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import get_project_root, to_sql_path
from duckdb_utils import get_connection
from validation import CheckSuite
from bootstrap_log import append_log, make_entry

PROJECT_ROOT = get_project_root()
con          = get_connection()  # in-memory — never writes to data/

LOG_PATH  = PROJECT_ROOT / "db" / "bootstrap_log.json"
GATE_PATH = PROJECT_ROOT / "db" / "landing_gate.json"

SOURCES = {
    "receita_federal": PROJECT_ROOT / "data" / "raw" / "receita_federal",
    "pncp"           : PROJECT_ROOT / "data" / "raw" / "pncp",
    "transparencia"  : PROJECT_ROOT / "data" / "raw" / "portal_transparencia",
    "compras_gov"    : PROJECT_ROOT / "data" / "raw" / "compras_gov",
}

SOURCE_ID = "landing_validate"

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"LOG_PATH     : {LOG_PATH}")
print(f"GATE_PATH    : {GATE_PATH}")
print()
for name, path in SOURCES.items():
    status = "EXISTS" if path.exists() else "MISSING"
    print(f"  {name:<22} {status}  {path}")


## Step 2 — Bootstrap log review

**Question:** Did the bootstrap scripts complete successfully for all sources?
Are there any ERROR or EMPTY periods that indicate incomplete downloads?


In [ ]:
if not LOG_PATH.exists():
    print("[WARN] bootstrap_log.json not found.")
    print("       Run bootstrap scripts (01-04) before this notebook.")
else:
    log_path_sql = to_sql_path(LOG_PATH)
    con.execute(
        f"CREATE OR REPLACE TABLE bootstrap_log AS "
        f"SELECT * FROM read_json_auto('{log_path_sql}')"
    )

    print("=== Bootstrap Log — Last Run Per Source ===")
    print()

    summary = con.execute("""
        SELECT
            source_id,
            COUNT(*) FILTER (WHERE status = 'SUCCESS') AS success,
            COUNT(*) FILTER (WHERE status = 'ERROR')   AS errors,
            COUNT(*) FILTER (WHERE status = 'EMPTY')   AS empty,
            COUNT(*) FILTER (WHERE status = 'SKIPPED') AS skipped,
            MAX(finished_at)                            AS last_run
        FROM bootstrap_log
        GROUP BY source_id
        ORDER BY source_id
    """).df()
    print(summary.to_string(index=False))

    errors = con.execute("""
        SELECT source_id, period, error_message
        FROM bootstrap_log
        WHERE status = 'ERROR'
        ORDER BY source_id, period
    """).df()
    if len(errors):
        print()
        print(f"[WARN] {len(errors)} ERROR entries found:")
        print(errors.to_string(index=False))
        print()
        print("Recommendation: re-run the affected bootstrap scripts before proceeding.")


## Step 3 — File inventory

**Question:** Are all expected files present for each source?
Are there any suspiciously small (empty) files that indicate a failed write?


In [ ]:
def inventory_source(name, path):
    """Inventory a source directory — returns status, file count, size, and empty files."""
    if not path.exists():
        return {"status": "MISSING", "files": 0, "total_mb": 0.0, "empty_files": []}
    all_files   = [f for f in path.rglob("*") if f.is_file()]
    empty_files = [f for f in all_files if f.stat().st_size <= 10]
    total_mb    = sum(f.stat().st_size for f in all_files) / (1024 * 1024)
    return {
        "status"     : "OK" if not empty_files else "WARN",
        "files"      : len(all_files),
        "total_mb"   : total_mb,
        "empty_files": [f.name for f in empty_files],
    }

print("=== File Inventory ===")
print()
print(f"  {'source':<22} {'status':<8} {'files':>6} {'total_mb':>10}")
print("  " + "-" * 52)

inventory = {}
for name, path in SOURCES.items():
    inv              = inventory_source(name, path)
    inventory[name]  = inv
    flag = " ← EMPTY FILES" if inv["empty_files"] else ""
    print(f"  {name:<22} {inv['status']:<8} {inv['files']:>6} {inv['total_mb']:>10.1f} MB{flag}")

total_files = sum(i["files"] for i in inventory.values())
total_mb    = sum(i["total_mb"] for i in inventory.values())
print("  " + "-" * 52)
print(f"  {'TOTAL':<22} {'':8} {total_files:>6} {total_mb:>10.1f} MB")

print()
for name, inv in inventory.items():
    if inv["empty_files"]:
        print(f"  [WARN] {name}: empty files → {inv['empty_files'][:5]}")


## Step 4 — Content sample

**Question:** Can each source actually be parsed by DuckDB?
Are ZIP files internally valid? Can JSON files be read without errors?


In [ ]:
def check_source_content(name: str, path: "Path", source_type: str) -> tuple[bool, str]:
    """
    Verify that a landing zone source is readable by DuckDB.

    Parameters
    ----------
    name        : source identifier key (e.g. 'pncp')
    path        : source root directory
    source_type : 'zip_dir' | 'json_flat' | 'json_fixed'
                  zip_dir   — RF: directory of monthly ZIP files
                  json_flat — PNCP/Compras: directory of monthly JSON files
                  json_fixed — Transparência: fixed filenames ceis.json + cnep.json

    Returns
    -------
    tuple[bool, str]
        (ok, status_message)
    """
    if not path.exists():
        return False, "MISSING"

    if source_type == "zip_dir":
        months = sorted([d for d in path.iterdir() if d.is_dir()])
        if not months:
            return False, "No month directories found"
        sample_dir = months[-1]
        zips       = sorted(sample_dir.glob("*.zip"))
        corrupted  = []
        for zp in zips:
            try:
                with zipfile.ZipFile(zp) as zf:
                    bad = zf.testzip()
                if bad:
                    corrupted.append(zp.name)
            except zipfile.BadZipFile:
                corrupted.append(zp.name)
        if corrupted:
            return False, f"Corrupted ZIPs: {corrupted}"
        return True, f"OK ({len(zips)} ZIPs in {sample_dir.name})"

    if source_type == "json_flat":
        files = sorted(path.glob("*.json"))
        if not files:
            return False, "No JSON files found"
        try:
            sql = to_sql_path(files[-1])
            n   = con.execute(
                f"SELECT COUNT(*) FROM read_json_auto('{sql}', maximum_object_size=33554432)"
            ).fetchone()[0]
            return (n > 0), f"OK — {files[-1].name}: {n:,} records" if n > 0 else "EMPTY"
        except Exception as exc:
            return False, f"ERROR parsing {files[-1].name}: {exc}"

    if source_type == "json_fixed":
        # Transparência: verify both ceis.json and cnep.json
        ok_all = True
        msgs   = []
        for fname in ["ceis.json", "cnep.json"]:
            fpath = path / fname
            if not fpath.exists():
                ok_all = False
                msgs.append(f"{fname}: MISSING")
                continue
            try:
                sql = to_sql_path(fpath)
                n   = con.execute(f"SELECT COUNT(*) FROM read_json_auto('{sql}')").fetchone()[0]
                ok  = n > 0
                ok_all = ok_all and ok
                msgs.append(f"{fname}: {'OK' if ok else 'EMPTY'} ({n:,} records)")
            except Exception as exc:
                ok_all = False
                msgs.append(f"{fname}: ERROR — {exc}")
        return ok_all, "  |  ".join(msgs)

    return False, f"Unknown source_type: {source_type}"


# Source type mapping
SOURCE_TYPES = {
    "receita_federal": "zip_dir",
    "pncp"           : "json_flat",
    "transparencia"  : "json_fixed",
    "compras_gov"    : "json_flat",
}

print("=== Content Sample ===")
print()

content_results = {}
for name, path in SOURCES.items():
    ok, msg = check_source_content(name, path, SOURCE_TYPES[name])
    content_results[name] = ok
    status = "OK" if ok else "FAIL"
    print(f"  [{status}] {name:<22} {msg}")


## Step 5 — Validation gate

**Question:** Does the landing zone pass all quality checks?
If not, what exactly failed and what action is needed?


In [ ]:
# Guard — ensure Steps 3 and 4 have run in this kernel session
assert "inventory" in dir(), "Run Step 3 (File inventory) before this step"
assert "content_results" in dir(), "Run Step 4 (Content sample) before this step"

# Capture validation start time here — not in Step 1 (kernel init)
STARTED_AT = datetime.now(timezone.utc).isoformat()

suite = CheckSuite("landing_validate")

# ── Presence checks ───────────────────────────────────────────────────────
for name in SOURCES:
    suite.add(
        f"{name} present",
        inventory[name]["status"] != "MISSING",
        inventory[name]["status"],
    )

# ── Empty file checks ─────────────────────────────────────────────────────
for name, inv in inventory.items():
    suite.add(
        f"{name} no empty files",
        len(inv["empty_files"]) == 0,
        f"{len(inv['empty_files'])} empty files" if inv["empty_files"] else "OK",
    )

# ── Content parseable checks ──────────────────────────────────────────────
for name, ok in content_results.items():
    suite.add(
        f"{name} content readable",
        ok,
        "OK" if ok else "FAILED — see Step 4",
    )

# ── Bootstrap log exists ──────────────────────────────────────────────────
suite.add(
    "bootstrap_log exists",
    LOG_PATH.exists(),
    str(LOG_PATH.exists()),
)

suite.report()


## Step 6 — Log result and write gate file

Records the validation result in `bootstrap_log.json` and writes `landing_gate.json`.
On failure: halts the kernel — Bronze notebooks must not run.


In [ ]:
FINISHED_AT = datetime.now(timezone.utc).isoformat()

failed_checks = [c.name for c in suite.failed_checks]
status        = "SUCCESS" if suite.all_passed else "ERROR"
error_msg     = f"Failed checks: {failed_checks}" if failed_checks else None

# ── Write gate file (read by Bronze notebooks in Step 1) ──────────────────
gate = {
    "status"        : status,
    "finished_at"   : FINISHED_AT,
    "failed_checks" : failed_checks,
}
GATE_PATH.parent.mkdir(parents=True, exist_ok=True)
GATE_PATH.write_text(json.dumps(gate, indent=2, ensure_ascii=False))
print(f"Gate file written: {GATE_PATH}")

# ── Log result to bootstrap_log.json ─────────────────────────────────────
entry = make_entry(
    source_id     = SOURCE_ID,
    period        = "pipeline",
    status        = status,
    started_at    = STARTED_AT,
    finished_at   = FINISHED_AT,
    error_message = error_msg,
)
append_log(LOG_PATH, entry)
print(f"Log entry written: source_id={SOURCE_ID}  status={status}")

# Close DuckDB connection — explicit is better than implicit
con.close()

# ── Gate decision ─────────────────────────────────────────────────────────
print()
if suite.all_passed:
    print("=" * 60)
    print("  ✅  LANDING GATE: PASSED — proceed to Bronze notebooks")
    print("=" * 60)
else:
    print("=" * 60)
    print("  ❌  LANDING GATE: FAILED — do NOT run Bronze notebooks")
    print("=" * 60)
    print()
    print("  Failed checks:")
    for name in failed_checks:
        print(f"    • {name}")
    print()
    print("  Action required:")
    print("    1. Fix the issues listed above")
    print("    2. Re-run the affected bootstrap script (01-04)")
    print("    3. Re-run this notebook")
    print()
    print("  Future enhancement: add webhook notification here")
    print("  (Slack/Teams/email) for scheduled pipelines with SLA.")
    print()
    # On Databricks replace with: dbutils.notebook.exit(json.dumps(gate))
    raise AssertionError(f"Landing gate FAILED: {failed_checks}")


## Step 7 — Dynamic summary

At-a-glance view of the landing zone state after validation.


In [ ]:
from IPython.display import display, Markdown

pncp_files    = len(list(SOURCES["pncp"].glob("*.json")))          if SOURCES["pncp"].exists()    else 0
compras_files = len(list(SOURCES["compras_gov"].glob("*.json")))   if SOURCES["compras_gov"].exists() else 0
rf_months     = len([d for d in SOURCES["receita_federal"].iterdir() if d.is_dir()])                 if SOURCES["receita_federal"].exists() else 0

total_files = sum(i["files"] for i in inventory.values())
total_mb    = sum(i["total_mb"] for i in inventory.values())

rows = ""
for name, inv in inventory.items():
    extra = ""
    if content_results.get(name) is False:
        extra = " — ❌ not readable"
    elif inv["empty_files"]:
        extra = f" — ⚠️ {len(inv['empty_files'])} empty"
    rows += f"| {name} | {inv['status']} | {inv['files']} | {inv['total_mb']:.1f} MB{extra} |\n"

summary = f"""
## Landing Zone — Validation Summary

**Result:** {'✅ PASSED — proceed to Bronze' if suite.all_passed else '❌ FAILED — fix issues above'}
**Timestamp:** {FINISHED_AT}

| Source | Status | Files | Size |
|---|---|---|---|
{rows}
**Total:** {total_files:,} files — {total_mb:.1f} MB

### Next step
{'Run `06_bronze_compras.ipynb`' if suite.all_passed else 'Fix failing sources and re-run this notebook.'}
"""

display(Markdown(summary))
